In [0]:
display(spark.sql("""
    SELECT *
    FROM banking_workspace.gold_dev.daily_transaction_summary
    ORDER BY business_date, currency
"""))

In [0]:
display(spark.sql("""
    SELECT *
    FROM banking_workspace.gold_dev.account_transaction_summary
    WHERE account_id = 'A001'
    ORDER BY business_date, currency
"""))

In [0]:
display(spark.sql("""
    SELECT *
    FROM banking_workspace.gold_dev.customer_transaction_summary
    ORDER BY customer_id, business_date, currency
"""))

In [0]:
display(spark.sql("""
    WITH expected AS (
        SELECT
            business_date,
            currency,
            COUNT(*) AS transaction_count,
            SUM(
                CASE
                    WHEN status = 'SUCCESS' AND direction = 'CREDIT'
                    THEN amount ELSE 0
                END
            ) AS total_credit,
            SUM(
                CASE
                    WHEN status = 'SUCCESS' AND direction = 'DEBIT'
                    THEN amount ELSE 0
                END
            ) AS total_debit
        FROM banking_workspace.silver_dev.transactions_current
        GROUP BY business_date, currency
    )
    SELECT
        COALESCE(s.business_date, g.business_date) AS business_date,
        COALESCE(s.currency, g.currency) AS currency,
        s.transaction_count AS silver_count,
        g.transaction_count AS gold_count,
        s.total_credit AS silver_credit,
        g.total_credit AS gold_credit,
        s.total_debit AS silver_debit,
        g.total_debit AS gold_debit
    FROM expected s
    FULL OUTER JOIN
        banking_workspace.gold_dev.daily_transaction_summary g
        ON s.business_date <=> g.business_date
        AND s.currency <=> g.currency
    WHERE NOT (s.transaction_count <=> g.transaction_count)
       OR NOT (s.total_credit <=> g.total_credit)
       OR NOT (s.total_debit <=> g.total_debit)
"""))